# 04 - Neural Network Model

**Project:** Predictive Modeling for Drug Discovery via Virtual Screening  
**Student:** Milica Jeftic (ID: 89211255)  
**Date:** January 2026  
**Dataset:** Kaggle - Drug Discovery Virtual Screening Dataset

---

## Goal of This Notebook

This notebook implements a feed-forward neural network for the compound activity classification task. The project proposal included a neural network model to test whether a deeper non-linear model can learn useful relationships between molecular/protein descriptors and biological activity.

The model is implemented with scikit-learn's `MLPClassifier`, which is a multi-layer feed-forward neural network. This keeps the notebook runnable in the project `.conda` environment while still satisfying the neural network modeling requirement.

---

## Expected Outputs

- Neural network validation and test metrics
- Training loss and validation score curves
- Confusion matrix and ROC curve visualization
- Saved neural network model in `models/`
- Saved metrics table in `results/metrics/`


## 1. Environment Setup

This section imports the required libraries, configures reproducibility, and defines project paths.

In [ ]:
# ============================
# Environment & Configuration
# ============================

import os
import sys
import random
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.utils.class_weight import compute_sample_weight

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# ----------------------------
# Reproducibility & Warnings
# ----------------------------
RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ----------------------------
# Pandas display options
# ----------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

# ----------------------------
# Visualization defaults
# ----------------------------
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
rcParams["figure.figsize"] = (12, 6)
rcParams["font.size"] = 12

%matplotlib inline

# ----------------------------
# Project paths
# ----------------------------
def find_project_root(start_path):
    """Find the project root from either the repository root or the notebooks directory."""
    current_path = os.path.abspath(start_path)
    for _ in range(3):
        expected_items = [
            os.path.join(current_path, "data"),
            os.path.join(current_path, "notebooks"),
            os.path.join(current_path, "README.md"),
        ]
        if all(os.path.exists(path) for path in expected_items):
            return current_path
        current_path = os.path.dirname(current_path)
    raise FileNotFoundError("Could not locate the project root directory.")

PROJECT_ROOT = find_project_root(os.getcwd())
DATA_PROCESSED_PATH = os.path.join(PROJECT_ROOT, "data", "processed")
MODELS_PATH = os.path.join(PROJECT_ROOT, "models")
RESULTS_PATH = os.path.join(PROJECT_ROOT, "results")

print("=" * 60)
print("Environment initialized successfully")
print("=" * 60)
print(f"Python       : {sys.version.split()[0]}")
print(f"Numpy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Scikit-learn : {__import__('sklearn').__version__}")
print("-" * 60)
print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_PROCESSED_PATH}")
print(f"Models dir   : {MODELS_PATH}")
print("=" * 60)


## 2. Load Preprocessed Data

The neural network uses the same scaled train, validation, and test sets produced in notebook 01. No additional preprocessing is performed here.

In [ ]:
print("=" * 60)
print("LOADING PREPROCESSED DATA")
print("=" * 60)

X_train = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "X_train.csv"))
X_val = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "X_val.csv"))
X_test = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "X_test.csv"))

y_train = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "y_train.csv")).squeeze("columns").astype(int)
y_val = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "y_val.csv")).squeeze("columns").astype(int)
y_test = pd.read_csv(os.path.join(DATA_PROCESSED_PATH, "y_test.csv")).squeeze("columns").astype(int)

assert len(X_train) == len(y_train), "Train X/y length mismatch"
assert len(X_val) == len(y_val), "Validation X/y length mismatch"
assert len(X_test) == len(y_test), "Test X/y length mismatch"
assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns), "Feature columns do not match across splits"

feature_cols = list(X_train.columns)
input_dim = X_train.shape[1]

print("Data loaded successfully")
print(f"Training set   : X={X_train.shape}, y={y_train.shape}")
print(f"Validation set : X={X_val.shape}, y={y_val.shape}")
print(f"Test set       : X={X_test.shape}, y={y_test.shape}")
print(f"Input dimension: {input_dim}")

print()
print("Class proportions:")
for split_name, split_y in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{split_name}: {split_y.value_counts(normalize=True).sort_index().to_dict()}")


## 3. Model Definition

The neural network is a small multi-layer perceptron with two hidden layers. L2 regularization is controlled by `alpha`, and `early_stopping=True` stops training when the model no longer improves on an internal validation split from the training data.

In [ ]:
nn_model = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    alpha=0.001,
    batch_size=32,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=RANDOM_STATE,
)

print("Neural network model initialized")
print(f"Hidden layers       : {nn_model.hidden_layer_sizes}")
print(f"Activation          : {nn_model.activation}")
print(f"Solver              : {nn_model.solver}")
print(f"L2 alpha            : {nn_model.alpha}")
print(f"Early stopping      : {nn_model.early_stopping}")
print(f"Max iterations      : {nn_model.max_iter}")


## 4. Training

Balanced sample weights are computed from the training labels to account for the moderate 70/30 class imbalance. The model is trained only on the training split.

In [ ]:
print("=" * 60)
print("TRAINING NEURAL NETWORK")
print("=" * 60)

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

nn_model.fit(X_train, y_train, sample_weight=sample_weights)

print("Training completed")
print(f"Iterations run: {nn_model.n_iter_}")
print(f"Final training loss: {nn_model.loss_:.4f}")
if hasattr(nn_model, "best_validation_score_"):
    print(f"Best internal validation score: {nn_model.best_validation_score_:.4f}")


## 5. Learning Curves

The training loss curve and internal validation score curve are used to inspect convergence. The external validation set is still kept separate for model evaluation.

In [ ]:
figures_path = os.path.join(RESULTS_PATH, "figures")
os.makedirs(figures_path, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(np.arange(1, len(nn_model.loss_curve_) + 1), nn_model.loss_curve_, label="Training loss")
axes[0].set_title("Neural Network Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

if hasattr(nn_model, "validation_scores_") and nn_model.validation_scores_:
    axes[1].plot(
        np.arange(1, len(nn_model.validation_scores_) + 1),
        nn_model.validation_scores_,
        label="Internal validation accuracy",
    )
    axes[1].set_ylabel("Accuracy")
else:
    axes[1].text(0.5, 0.5, "No validation scores available", ha="center", va="center")

axes[1].set_title("Neural Network Internal Validation Score")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
learning_curve_path = os.path.join(figures_path, "neural_network_learning_curves.png")
plt.savefig(learning_curve_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Learning curves saved to: {learning_curve_path}")


## 6. Validation and Test Evaluation

The neural network is evaluated using the same metrics as the previous notebooks: accuracy, precision, recall, F1-score, and ROC-AUC.

In [ ]:
def evaluate_neural_network(model, X, y, split_name):
    """Evaluate the neural network and return metrics plus predictions."""
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = model.predict(X)

    metrics = {
        "model": "Neural Network",
        "split": split_name,
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred),
        "recall": recall_score(y, y_pred),
        "f1_score": f1_score(y, y_pred),
        "roc_auc": roc_auc_score(y, y_proba),
    }
    return metrics, y_pred, y_proba

val_metrics, y_val_pred, y_val_proba = evaluate_neural_network(nn_model, X_val, y_val, "validation")
test_metrics, y_test_pred, y_test_proba = evaluate_neural_network(nn_model, X_test, y_test, "test")

nn_metrics_df = pd.DataFrame([val_metrics, test_metrics])
display(nn_metrics_df)


## 7. Visual Evaluation

The confusion matrix and ROC curve summarize neural network performance on the held-out test set.

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
test_auc = roc_auc_score(y_test, y_test_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    ax=axes[0],
    cbar=False,
    xticklabels=["Inactive", "Active"],
    yticklabels=["Inactive", "Active"],
)
axes[0].set_title("Confusion Matrix - Neural Network Test Set", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

axes[1].plot(fpr, tpr, linewidth=2, label=f"ROC-AUC = {test_auc:.4f}")
axes[1].plot([0, 1], [0, 1], linestyle="--", linewidth=1, label="Random Classifier")
axes[1].set_title("ROC Curve - Neural Network Test Set", fontsize=12, fontweight="bold")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(loc="lower right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
test_plot_path = os.path.join(figures_path, "neural_network_test_evaluation.png")
plt.savefig(test_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Neural network test visualization saved to: {test_plot_path}")


## 8. Save Model and Metrics

The trained neural network and metrics table are saved for final comparison in notebook 05.

In [ ]:
print("=" * 60)
print("SAVING NEURAL NETWORK OUTPUTS")
print("=" * 60)

os.makedirs(MODELS_PATH, exist_ok=True)
metrics_path = os.path.join(RESULTS_PATH, "metrics")
os.makedirs(metrics_path, exist_ok=True)

nn_model_path = os.path.join(MODELS_PATH, "neural_network_mlp.joblib")
nn_metrics_path = os.path.join(metrics_path, "neural_network_metrics.csv")

joblib.dump(nn_model, nn_model_path)
nn_metrics_df.to_csv(nn_metrics_path, index=False)

print(f"Neural network model saved to: {nn_model_path}")
print(f"Neural network metrics saved to: {nn_metrics_path}")
display(nn_metrics_df)


## 9. Neural Network Summary

This notebook trained a feed-forward neural network as the neural network model proposed for the project. The implementation uses scikit-learn's `MLPClassifier`, which is available in the project `.conda` environment and represents a fully connected neural network model.

The model uses the same processed train/validation/test split as the previous notebooks, balanced sample weights for class imbalance, L2 regularization, and early stopping.

### Key Results

The neural network achieved strong performance:

- Validation accuracy: 0.9745
- Validation F1-score: 0.9586
- Validation ROC-AUC: 0.9982
- Test accuracy: 0.9854
- Test F1-score: 0.9762
- Test ROC-AUC: 0.9993

These results are close to the Logistic Regression baseline, but they do not exceed the tree-based models from notebook 03, which achieved perfect scores on this dataset.

### Interpretation

The neural network confirms that the processed descriptors contain strong signal for predicting compound activity. However, the tree-based models remain the best-performing models on this dataset. Since feature importance analysis showed that `binding_affinity` is highly dominant, the neural network mainly serves as an additional model-family comparison rather than a practical improvement over the tree-based models.

The saved neural network model and metrics will be included in the final model comparison notebook.
